In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/exoplanets_raw.csv', comment='#')

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()

/tmp/ipykernel_20265/641057413.py:1: DtypeWarning: Columns (12,13,37,38,52,53,54,55,70,71,78,79,81,94,95,96,97) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/exoplanets_raw.csv', comment='#')


Rows: 39443
Columns: 100


,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,pl_pubdate,releasedate,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99
0,11 Com b,11 Com,1,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2023-08,2023-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2008-01,2014-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2011-08,2014-07-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,2009-10,2014-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11 UMi b,11 UMi,1,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,2017-03,2018-09-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
useful_columns = [
    'pl_name', 'disc_year', 'discoverymethod',
    'pl_rade', 'pl_bmasse', 'pl_orbper',
    'sy_dist', 'pl_eqt', 'st_teff', 'sy_snum'
]

df = df[useful_columns]
print(df.shape)
df.head()

(39443, 10)


,pl_name,disc_year,discoverymethod,pl_rade,pl_bmasse,pl_orbper,sy_dist,pl_eqt,st_teff,sy_snum
0,11 Com b,2007.0,Radial Velocity,NaN,4914.898486,323.21,93.1846,NaN,4874,2
1,11 Com b,2007.0,Radial Velocity,NaN,6165.600000,326.03,93.1846,NaN,4742,2
2,11 Com b,2007.0,Radial Velocity,NaN,5434.700000,NaN,93.1846,NaN,NaN,2
3,11 UMi b,2009.0,Radial Velocity,NaN,3337.070000,516.22,125.321,NaN,4340,1
4,11 UMi b,2009.0,Radial Velocity,NaN,4684.814200,516.21997,125.321,NaN,4213,1


In [4]:
df = df.drop_duplicates(subset='pl_name', keep='first')

print(f"Unique planets: {df.shape[0]}")

Unique planets: 6128


In [5]:
df.isnull().sum()

pl_name               0
disc_year             1
discoverymethod       0
pl_rade            2400
pl_bmasse          3770
pl_orbper           563
sy_dist             128
pl_eqt             3415
st_teff            1047
sy_snum               0
dtype: int64

In [6]:
df.describe()

,disc_year,pl_rade,pl_bmasse,pl_eqt,sy_snum
count,6127.000000,3728.000000,2358.000000,2713.000000,6128.000000
mean,2016.954464,5.274184,825.025167,846.276786,1.103296
std,4.945150,70.275857,1659.862590,489.588319,0.342233
min,1992.000000,-0.009000,-0.017040,-35.000000,1.000000
25%,2014.000000,1.439599,8.902500,516.000000,1.000000
50%,2016.000000,2.300000,162.092488,779.150000,1.000000
75%,2021.000000,3.710000,794.553263,1134.310000,1.000000
max,2026.000000,4282.980000,22934.497849,3053.000000,4.000000


In [7]:
df.to_csv('../data/exoplanets_clean.csv', index=False)
print("File saved!")

File saved!


In [9]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/exoplanets.db')
df.to_sql('exoplanets', engine, if_exists='replace', index=False)

print("Database created!")

Database created!


In [11]:
import pandas as pd

# Quel est la méthode de détection la plus utilisée ?
# Combien de planètes par méthode de détection ?
query = """
SELECT discoverymethod, COUNT(*) as total
FROM exoplanets
GROUP BY discoverymethod
ORDER BY total DESC
"""

result = pd.read_sql(query, engine)
result

,discoverymethod,total
0,Transit,4511
1,Radial Velocity,1171
2,Microlensing,270
3,Imaging,94
4,Transit Timing Variations,39
5,Eclipse Timing Variations,17
6,Orbital Brightness Modulation,9
7,Pulsar Timing,8
8,Astrometry,6
9,Pulsation Timing Variations,2


In [12]:
# Découverte par année
query = """
SELECT disc_year, COUNT(*) as total
FROM exoplanets
WHERE disc_year IS NOT NULL
GROUP BY disc_year
ORDER BY disc_year ASC
"""

result2 = pd.read_sql(query, engine)
result2

,disc_year,total
0,1992.0,2
1,1994.0,1
2,1995.0,1
3,1996.0,6
4,1997.0,1
5,1998.0,6
6,1999.0,13
7,2000.0,16
8,2001.0,12
9,2002.0,29


In [17]:
# Corriger le type de sy_dist
df['sy_dist'] = pd.to_numeric(df['sy_dist'], errors='coerce')

# Remettre à jour la base SQLite
df.to_sql('exoplanets', engine, if_exists='replace', index=False)
print("Database updated!")

Database updated!


In [20]:
# The 10 nearest planets from Earth
query = """
SELECT pl_name, discoverymethod, sy_dist, disc_year
FROM exoplanets
WHERE sy_dist > 0
ORDER BY sy_dist ASC
LIMIT 10
"""

result3 = pd.read_sql(query, engine)
result3

,pl_name,discoverymethod,sy_dist,disc_year
0,WASP-151 b,Transit,0.306690,2017.0
1,Proxima Cen b,Radial Velocity,1.301190,2016.0
2,Proxima Cen d,Radial Velocity,1.301190,2025.0
3,Barnard b,Radial Velocity,1.826550,2024.0
4,Barnard c,Radial Velocity,1.826550,2025.0
5,Barnard d,Radial Velocity,1.826550,2025.0
6,Barnard e,Radial Velocity,1.826550,2025.0
7,WASP-82 b,Transit,1.893836,2015.0
8,WASP-24 b,Transit,2.343286,2010.0
9,HAT-P-64 b,Transit,2.431273,2021.0
